# 03. 모델링
Netflix 고객 이탈 예측 — ML/DL 모델 비교 및 최적 모델 선택

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 전처리된 데이터 로드

In [ ]:
# 트리 모델용 (Label Encoding)
X_train_label = pd.read_csv('../data/processed/X_train_label.csv')
X_test_label = pd.read_csv('../data/processed/X_test_label.csv')

# LR, SVM, MLP용 (One-Hot + Scaled)
X_train_ohe = pd.read_csv('../data/processed/X_train_ohe_scaled.csv')
X_test_ohe = pd.read_csv('../data/processed/X_test_ohe_scaled.csv')

# 타겟
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(f'Label 버전 - Train: {X_train_label.shape}, Test: {X_test_label.shape}')
print(f'OHE 버전   - Train: {X_train_ohe.shape}, Test: {X_test_ohe.shape}')
print(f'타겟       - Train: {y_train.shape}, Test: {y_test.shape}')

NameError: name 'pd' is not defined

## 2. ML 모델 학습 및 비교

| 순서 | 모델 | 역할 |
|---|---|---|
| 1 | Logistic Regression | Baseline (기준선) |
| 2 | Decision Tree | 해석용 (이탈 경로 시각화) |
| 3 | Random Forest | 안정적 성능 |
| 4 | XGBoost | 성능 경쟁자 1 |
| 5 | LightGBM | 성능 경쟁자 2 |
| 6 | CatBoost | 범주형 특화 |
| 7 | SVM | 다른 방식으로 다양성 확보 |

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import cross_val_score

# 모델별 데이터 매핑
# OHE+Scaled: LR, SVM
# Label: DT, RF, GBM, XGBoost, LightGBM
# CatBoost: Label (자체 범주 처리)
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), 'ohe'),
    'Decision Tree': (DecisionTreeClassifier(random_state=42), 'label'),
    'Random Forest': (RandomForestClassifier(n_estimators=100, random_state=42), 'label'),
    'Gradient Boosting': (GradientBoostingClassifier(n_estimators=100, random_state=42), 'label'),
    'XGBoost': (XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'), 'label'),
    'LightGBM': (LGBMClassifier(n_estimators=100, random_state=42, verbose=-1), 'label'),
    'CatBoost': (CatBoostClassifier(n_estimators=100, random_state=42, verbose=0), 'label'),
    'SVM': (SVC(random_state=42, probability=True), 'ohe'),
}

results = []
trained_models = {}

for name, (model, data_type) in models.items():
    X_tr = X_train_ohe if data_type == 'ohe' else X_train_label
    X_te = X_test_ohe if data_type == 'ohe' else X_test_label
    
    # 학습
    model.fit(X_tr, y_train)
    trained_models[name] = (model, data_type)
    
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    
    # 교차검증
    cv_scores = cross_val_score(model, X_tr, y_train, cv=5, scoring='f1')
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'CV F1 (mean)': cv_scores.mean(),
        'CV F1 (std)': cv_scores.std(),
    })
    print(f'{name:25s} | F1: {results[-1]["F1"]:.4f} | AUC: {results[-1]["ROC-AUC"]:.4f} | CV: {cv_scores.mean():.4f}±{cv_scores.std():.4f}')

df_results = pd.DataFrame(results).sort_values('F1', ascending=False)
df_results

## 3. 모델 성능 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# F1 Score 비교
df_sorted = df_results.sort_values('F1', ascending=True)
colors = ['#e74c3c' if v == df_sorted['F1'].max() else '#3498db' for v in df_sorted['F1']]
axes[0].barh(df_sorted['Model'], df_sorted['F1'], color=colors)
for i, v in enumerate(df_sorted['F1']):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center')
axes[0].set_title('모델별 F1 Score', fontsize=13)
axes[0].set_xlabel('F1 Score')

# 전체 지표 비교
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
df_plot = df_results.set_index('Model')[metrics].sort_values('F1', ascending=True)
df_plot.plot(kind='barh', ax=axes[1], width=0.7)
axes[1].set_title('모델별 전체 지표 비교', fontsize=13)
axes[1].set_xlabel('Score')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 4. ROC Curve

In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8, 8))

for name, (model, data_type) in trained_models.items():
    X_te = X_test_ohe if data_type == 'ohe' else X_test_label
    y_prob = model.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve - 전체 모델 비교', fontsize=14)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 5. Confusion Matrix (상위 3개 모델)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

top3 = df_results.head(3)['Model'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3):
    model, data_type = trained_models[name]
    X_te = X_test_ohe if data_type == 'ohe' else X_test_label
    y_pred = model.predict(X_te)
    
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Stayed', 'Churned'])
    disp.plot(ax=axes[i], cmap='Blues')
    axes[i].set_title(f'{name}', fontsize=12)

plt.suptitle('상위 3개 모델 Confusion Matrix', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
for name in top3:
    model, data_type = trained_models[name]
    X_te = X_test_ohe if data_type == 'ohe' else X_test_label
    y_pred = model.predict(X_te)
    
    print(f'\n=== {name} ===')
    print(classification_report(y_test, y_pred, target_names=['Stayed', 'Churned']))

## 6. Feature Importance

In [ ]:
# 트리 기반 상위 2개 모델
tree_names = [n for n, (m, _) in trained_models.items() if hasattr(m, 'feature_importances_')]
top_tree = df_results[df_results['Model'].isin(tree_names)].head(2)['Model'].tolist()

fig, axes = plt.subplots(1, len(top_tree), figsize=(8 * len(top_tree), 6))
if len(top_tree) == 1:
    axes = [axes]

for i, name in enumerate(top_tree):
    model, _ = trained_models[name]
    importance = pd.Series(model.feature_importances_, index=X_train_label.columns)
    importance = importance.sort_values(ascending=True)
    
    importance.plot(kind='barh', ax=axes[i], color='#2ecc71')
    axes[i].set_title(f'{name} — Feature Importance', fontsize=12)
    axes[i].set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 7. 하이퍼파라미터 튜닝 (최적 모델)

In [ ]:
from sklearn.model_selection import GridSearchCV

best_model_name = top_tree[0]
best_model_obj, best_data_type = trained_models[best_model_name]
X_tr = X_train_ohe if best_data_type == 'ohe' else X_train_label
X_te = X_test_ohe if best_data_type == 'ohe' else X_test_label

print(f'튜닝 대상: {best_model_name}')

param_grids = {
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10],
    },
    'XGBoost': {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
    },
    'LightGBM': {
        'n_estimators': [100, 200, 300],
        'max_depth': [-1, 10, 20],
        'learning_rate': [0.01, 0.1, 0.2],
    },
    'CatBoost': {
        'iterations': [100, 200, 300],
        'depth': [4, 6, 8],
        'learning_rate': [0.01, 0.1, 0.2],
    },
    'Gradient Boosting': {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
    },
    'Decision Tree': {
        'max_depth': [None, 5, 10, 20],
        'min_samples_split': [2, 5, 10, 20],
        'min_samples_leaf': [1, 2, 5],
    },
}

param_grid = param_grids.get(best_model_name, {})

grid_search = GridSearchCV(
    best_model_obj, param_grid,
    cv=5, scoring='f1', n_jobs=-1, verbose=1
)
grid_search.fit(X_tr, y_train)

print(f'\n최적 파라미터: {grid_search.best_params_}')
print(f'최적 CV F1: {grid_search.best_score_:.4f}')

In [ ]:
# 튜닝 전 vs 후 비교
best_tuned = grid_search.best_estimator_
y_pred_tuned = best_tuned.predict(X_te)
y_pred_before = best_model_obj.predict(X_te)

print(f'=== {best_model_name} 튜닝 전 vs 후 ===')
print(f'{"지표":15s} | {"튜닝 전":>10s} | {"튜닝 후":>10s}')
print('-' * 42)
for metric_name, metric_fn in [('Accuracy', accuracy_score), ('Precision', precision_score),
                                ('Recall', recall_score), ('F1', f1_score)]:
    before = metric_fn(y_test, y_pred_before)
    after = metric_fn(y_test, y_pred_tuned)
    diff = after - before
    print(f'{metric_name:15s} | {before:10.4f} | {after:10.4f} ({diff:+.4f})')

In [ ]:
# 튜닝 후 Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_tuned)
disp = ConfusionMatrixDisplay(cm, display_labels=['Stayed', 'Churned'])
disp.plot(ax=ax, cmap='Blues')
ax.set_title(f'{best_model_name} (튜닝 후)', fontsize=13)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_tuned, target_names=['Stayed', 'Churned']))

## 8. 최종 모델 저장

In [ ]:
import os

os.makedirs('../models', exist_ok=True)

joblib.dump(best_tuned, '../models/best_model.pkl')

print(f'최종 모델: {best_model_name}')
print(f'최적 파라미터: {grid_search.best_params_}')
print(f'\n저장된 파일:')
for f in os.listdir('../models'):
    size = os.path.getsize(f'../models/{f}') / 1024
    print(f'  {f}: {size:.0f} KB')

## 9. 최종 정리

### 모델 선택 근거
- F1 Score, ROC-AUC, 교차검증 안정성을 종합적으로 고려
- Confusion Matrix에서 이탈 고객(Churned)에 대한 Recall이 높은 모델 우선
  - 비즈니스 관점: 이탈 고객을 놓치는 것(FN)이 비이탈 고객을 잘못 분류하는 것(FP)보다 비용이 큼

### Feature Importance 해석
- 어떤 피처가 이탈에 가장 큰 영향을 미치는지 위 차트 참고
- 비즈니스 액션 제안에 활용 가능

### 다음 단계
- SHAP 분석으로 더 깊은 해석
- MLP (딥러닝) 모델 비교
- Streamlit 기반 예측 서비스 배포 (app/app.py)